# Chat via LangGraph

In [9]:
import { START, END, MessagesAnnotation, StateGraph, MemorySaver } from "@langchain/langgraph";
import { Annotations } from "@common/annotations.ts";
import { readFileSync } from 'node:fs';

import { PROMPT } from '../../server/src/Agents/Elements/assessment/prompt.ts';
import { Logger, getLogger } from '../../server/src/Logger.ts';
import { newModel } from '../../server/src/Agents/Agent.ts';
import { makeUserPrompt } from '../../server/src/Agents/Elements/annotation/annotate.ts';

In [10]:
const logger = getLogger();
const llm = newModel("Anthropic");

// Define the function that calls the model
const callModel = async (state: typeof MessagesAnnotation.State) => {
  const response = await llm.invoke(state.messages);
  // Update message history with response:
  return { messages: response };
};

// Define a new graph
const workflow = new StateGraph(MessagesAnnotation)
  // Define the (single) node in the graph
  .addNode("model", callModel)
  .addEdge(START, "model")
  .addEdge("model", END);

// Add memory
const memory = new MemorySaver();
const app = workflow.compile({ checkpointer: memory });

In [11]:
const userUUID: string = "0";
const lhsText = readFileSync('../../frontend/public/data/AES/selected-text.txt', 'utf-8');
const rhsText = readFileSync('../../frontend/public/data/AES/pre-written.txt', 'utf-8');
const currentAnnotations: Annotations = { "mappings": [], "lhsLabels": [], "rhsLabels": [] };
const resetChat = true;
const userInput: string = "Does the Lean specification define all relevant invariants and preconditions as described in the requirements?";

In [12]:
const config = { configurable: { thread_id: userUUID } };
const userContext = makeUserPrompt(lhsText, rhsText, currentAnnotations, logger);
const chatPrompt = PROMPT + userContext;
const systemInput = { 
  role: "system",
  content: chatPrompt
};
const input : (typeof systemInput)[] = [];

logger.info(`Reset chat? ${resetChat}`);
if (resetChat) {
  input.push(systemInput);
} 

logger.info(`Chat Prompt:\n${chatPrompt}`);
input.push({ 
  role: "user",
  content: userInput
});
logger.info(`Full input:\n${JSON.stringify(input, null, 2)}`);
logger.info("Sending prompt to Claude.");

const output = await app.invoke({ messages: input }, config);
const res = output.messages[output.messages.length - 1]
logger.info("Received response from Claude.");

console.log(res.content);

To determine if the Lean specification defines all relevant invariants and preconditions as described in the requirements, I will go through the requirements and the Lean code and make the following comparisons:

1. **Key Length, Block Size, and Number of Rounds**:
   - The requirements specify the key length, block size, and number of rounds for AES-128, AES-192, and AES-256 in Table 3.
   - The Lean code defines these parameters in the `AES` namespace, with `Nk` representing the number of 32-bit words in the key, `Nb` representing the number of 32-bit words in the block (which is fixed at 4), and `Nr` representing the number of rounds.
   - The Lean code correctly implements the values specified in the requirements.

2. **Input and Output Formats**:
   - The requirements state that the input to `CIPHER()` is a 16-byte block, and the output is also a 16-byte block.
   - The Lean code uses `Array UInt8` to represent the input, output, and state, which matches the 16-byte block size spe